# RAG Agent validation (dev)
Este notebook es utilizado para realizar la validación del RAG antes de hacer el deploy en el serving endpoint

> Nota: en Compute Serverless suele faltar `mlflow`; por eso instalamos dependencias en la primera celda.


In [0]:
# # # # ==============================
# # # # 1) Dependencias (solo para pruebas locales)
# # # # ==============================
# # # # En Serverless compute puede no venir mlflow instalado.
%pip install -U mlflow
%pip install -U databricks-vectorsearch mlflow databricks-sdk
dbutils.library.restartPython()


In [0]:
# ==============================
# 2) Config (ajustar a tu workspace)
# ==============================
from pathlib import Path
import time

# --- Vector Search ---
VS_ENDPOINT = "bind_agent_vs"
VS_INDEX_FULL_NAME = "bind_agent.docs.pdf_chunks_vs_idx"

# --- Endpoints ---
EMBED_ENDPOINT = "databricks-bge-large-en"  # for query_vector
EMB_ENDPOINT = EMBED_ENDPOINT  # backward-compatible alias
LLM_ENDPOINT = "databricks-gemma-3-12b"
# LLM_ENDPOINT = _get_env("RAG_LLM_ENDPOINT", default="databricks-gpt-5-2")

# --- Retrieval params ---
TOP_K_CANDIDATES = "40"
LEX_FALLBACK_LIMIT = "20"
# TOP_K_CANDIDATES = "120"
# LEX_FALLBACK_LIMIT = "60"
TOP_K_FINAL = "8"

# --- Prompt/context limits ---
MAX_CONTEXT_CHARS = "14000"
RERANK_SNIPPET_CHARS = "1200"

# --- LLM params ---
TEMPERATURE_RERANK = "0.0"
TEMPERATURE_ANSWER = "0.2"
MAX_TOKENS_ANSWER = "900"

# --- Retry params ---
MAX_RETRIES = "3"
RETRY_SLEEP_SECS = "1.0"

# MLflow / UC
# Recomendación: catalog.schema.model
UC_MODEL_NAME = "bind_agent.docs.rag_agent"

# Experimento: usar ruta en /Users/... para evitar problemas de permisos
current_user = spark.sql("select current_user() as u").first()["u"]
EXPERIMENT_PATH = f"/Users/{current_user}/bind_agent/rag_agent_deploy"

# Serving endpoint para el agente
MODEL_SERVING_ENDPOINT = "bind_agent_rag_agent"


In [0]:
import os, importlib.util
from pathlib import Path

# 0) Setear env vars usadas en el RAG
os.environ["RAG_LLM_ENDPOINT"] = LLM_ENDPOINT
os.environ["RAG_EMBED_ENDPOINT"] = EMBED_ENDPOINT
os.environ["RAG_VS_ENDPOINT"] = VS_ENDPOINT
os.environ["RAG_VS_INDEX"] = VS_INDEX_FULL_NAME
os.environ["RAG_VS_INDEX_FULL_NAME"] = VS_INDEX_FULL_NAME
os.environ["RAG_TOP_K_CANDIDATES"] = TOP_K_CANDIDATES
os.environ["RAG_TOP_K_FINAL"] = TOP_K_FINAL
os.environ["RAG_LEX_FALLBACK_LIMIT"] = LEX_FALLBACK_LIMIT
os.environ["RAG_MAX_CONTEXT_CHARS"] = MAX_CONTEXT_CHARS
os.environ["RAG_RERANK_SNIPPET_CHARS"] = RERANK_SNIPPET_CHARS
os.environ["RAG_TEMPERATURE_RERANK"] = TEMPERATURE_RERANK
os.environ["RAG_TEMPERATURE_ANSWER"] = TEMPERATURE_ANSWER
os.environ["RAG_MAX_TOKENS_ANSWER"] = MAX_TOKENS_ANSWER
os.environ["RAG_MAX_RETRIES"] = MAX_RETRIES
os.environ["RAG_RETRY_SLEEP_SECS"] = RETRY_SLEEP_SECS
# os.environ["DATABRICKS_TOKEN"] = token # Acceso para el service endpoint al search index 
# os.environ["DATABRICKS_HOST"] = host # Acceso para el service endpoint al search index 

# env usadas en el serving endpoint
env_vars = {
    "RAG_LLM_ENDPOINT": os.environ["RAG_LLM_ENDPOINT"],
    "RAG_EMBED_ENDPOINT": os.environ["RAG_EMBED_ENDPOINT"],
    "RAG_VS_ENDPOINT": os.environ["RAG_VS_ENDPOINT"],
    "RAG_VS_INDEX": os.environ["RAG_VS_INDEX"],
    "RAG_VS_INDEX_FULL_NAME": os.environ["RAG_VS_INDEX_FULL_NAME"],
    "RAG_TOP_K_CANDIDATES": os.environ["RAG_TOP_K_CANDIDATES"],
    "RAG_TOP_K_FINAL": os.environ["RAG_TOP_K_FINAL"],
    "RAG_LEX_FALLBACK_LIMIT": os.environ["RAG_LEX_FALLBACK_LIMIT"],
    "RAG_MAX_CONTEXT_CHARS": os.environ["RAG_MAX_CONTEXT_CHARS"],
    "RAG_RERANK_SNIPPET_CHARS": os.environ["RAG_RERANK_SNIPPET_CHARS"],
    "RAG_TEMPERATURE_RERANK": os.environ["RAG_TEMPERATURE_RERANK"],
    "RAG_TEMPERATURE_ANSWER": os.environ["RAG_TEMPERATURE_ANSWER"],
    "RAG_MAX_TOKENS_ANSWER": os.environ["RAG_MAX_TOKENS_ANSWER"],
    "RAG_MAX_RETRIES": os.environ["RAG_MAX_RETRIES"],
    "RAG_RETRY_SLEEP_SECS": os.environ["RAG_RETRY_SLEEP_SECS"],
    # "DATABRICKS_TOKEN": os.environ["DATABRICKS_TOKEN"],
    # "DATABRICKS_HOST": os.environ["DATABRICKS_HOST"],
}

Reinicio del cache para que se tomen los cambios en el RAG

In [0]:
import sys, importlib

def unload_rag_modules():
    prefixes = ("rag_agent", "rag_core", "rag_lib")
    for m in list(sys.modules.keys()):
        if m in prefixes or any(m.startswith(p + ".") for p in prefixes):
            sys.modules.pop(m, None)
    importlib.invalidate_caches()

unload_rag_modules()

Variables para activar el trace

In [0]:
# # 1) Activar trace
os.environ["RAG_DEBUG_TRACE"] = "1"
os.environ["RAG_DEBUG_TRACE_ALL"] = "1"   # si querés todos los candidatos

Preguntas que parsa el smoketest

In [0]:
## Si no tiene comentarios al lado es porque se está contestando correctamente

# query = "¿Cuál es el dato de previsiones para octubre 2025?"
# query = "¿Cuáles son los valores de previsiones para octubre 2025 por segmento?"
# query = "¿Cuáles es el valor de previsiones para octubre 2025 para empresas?"
# query = "Cual es el retorno sobre activos para octubre de 2025?"
# query = "Cual es el retorno sobre patrimonio para octubre de 2025?"
# query = "Como ha sido la evolución del ROE?" #Mejorar obtención de info de gráficos
# query = "¿Como ha sido la evolución del ROE a lo largo del tiempo?"
# query = "Como ha sido la evolución del ROA?"

# query = "Cuál fue el resultado de Banco Industrial en junio 2025 y su diferencia respecto a Banco Santander?" #Respuesta: 36081 y 338457.
# query = "Cuál fue el resultado de Banco Industrial en julio 2025 y su diferencia respecto a Banco Santander?" #Respuesta: 36081 y 338457.
#query = "Cuál fueron los ingresos YTD julio 2025 del Banco Santander?" #Respuesta: 36081 y 338457.

# query = "Cuál fue la variación interanual de los depósitos en $ para octubre 2025?"

# ------- mis preguntas -------

# query = '¿Cuál fue la variación mensual de los gastos operativos en octubre 2025?'
# query = '¿Qué participación tienen los depósitos vista vs plazo en octubre 2025?'
# query = '¿Cómo evolucionó el costo del riesgo en 2025?'
# query = '¿Cómo evolucionaron las altas de clientes en septiembre y octubre 2025?'
# query = '¿Cuál fue el crecimiento interanual de los ingresos por servicios en octubre 2025?'
query = '¿Cuál es el exceso/defecto de encajes al cierre de octubre 2025?'

SMOKE TEST: Respuesta curada

In [0]:
import importlib.util
from pathlib import Path
from textwrap import shorten
import sys

# --- 1) Cargar el módulo ---
agent_py = Path("/Workspace/Users/emanuel.vieira@sunnydata.ai/bind_agent/src/rag/rag_agent.py")
sys.path.insert(0, str(agent_py.parent))  # para que importe rag_lib
print(sys.path[0])

spec = importlib.util.spec_from_file_location("rag_agent_module", str(agent_py))
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

# --- 2) Helpers de pretty print ---
def _pp_result(res: dict, max_hits: int = 5, max_chars: int = 280):
    print("\n" + "="*88)
    print("QUERY:", res.get("query",""))
    print("-"*88)
    print("ANSWER:\n")
    print(res.get("answer","").strip())
    print("-"*88)

    cands = res.get("retrieved_candidates") or []
    reranked = res.get("reranked_hits") or []

    print(f"Retrieved candidates: {len(cands)} | Reranked hits: {len(reranked)}")

    hits = reranked if reranked else cands
    if not hits:
        print("\n(No hay hits para mostrar)")
        return

    print(f"\nTOP {min(max_hits, len(hits))} EVIDENCIAS:")
    for i, h in enumerate(hits[:max_hits], 1):
        file_date = h.get("file_date") or ""
        path = h.get("path") or ""
        page = h.get("page_num")
        topic = h.get("topic") or ""
        cid = h.get("chunk_id") or ""

        loc = []
        if file_date: loc.append(file_date)
        if path: loc.append(path)
        if page is not None: loc.append(f"p.{page}")
        if topic: loc.append(f"topic={topic}")
        loc = " | ".join(loc)

        print(f"\n[{i}] {loc}")
        print(f"    chunk_id: {cid}")

def _pp_minimal(res: dict):
    # por si querés una versión ultra corta
    print("\n" + "="*88)
    print(res.get("answer","").strip())
    print("="*88)


In [0]:
res = mod.answer_with_rag(query)

# Elegí una:
_pp_result(res, max_hits=10, max_chars=500)
# _pp_minimal(res)

In [0]:
# # ============================================================
# # Batch test: preguntas -> RAG -> Delta table (answers_test)
# # ============================================================
# import json
# from pathlib import Path
# from pyspark.sql import Row

# TABLE_NAME = "bind_agent.docs.answers_test"

# QUESTIONS_FILE = None
# # Ejemplos:
# # QUESTIONS_FILE = "/Workspace/Users/emanuel.vieira@sunnydata.ai/bind_agent/src/preguntas.txt"
# QUESTIONS_FILE = "/Workspace/Users/emanuel.vieira@sunnydata.ai/bind_agent/src/questions.json"

# # Si no usás archivo, definí la lista acá:
# QUESTIONS_INLINE = [
#     "Cual es el retorno sobre activos para octubre de 2025?",
#     # "¿Cuál es el dato de previsiones para octubre 2025?",
# ]

# MAX_EVIDENCES_TO_STORE = 10  # top N evidencias a guardar

# # ---------------------------
# # Helpers
# # ---------------------------
# def _load_questions(path: str | None):
#     if not path:
#         return QUESTIONS_INLINE

#     # DBFS (dbfs:/...) o path montado (/dbfs/...)
#     if path.startswith("dbfs:/"):
#         with dbutils.fs.open(path, "r") as f:
#             raw = f.read()
#     elif path.startswith("/dbfs/"):
#         raw = Path(path).read_text(encoding="utf-8")
#     else:
#         # archivo local del driver (por ejemplo /Workspace/Users/...)
#         raw = Path(path).read_text(encoding="utf-8")

#     path_lower = path.lower()
#     if path_lower.endswith(".json"):
#         obj = json.loads(raw)
#         if isinstance(obj, dict) and "questions" in obj:
#             qs = obj["questions"]
#         else:
#             qs = obj
#         return [q.strip() for q in qs if str(q).strip()]
#     else:
#         # txt: 1 por línea (ignora líneas vacías)
#         return [line.strip() for line in raw.splitlines() if line.strip()]

# def _format_answer_like_pp_result(res: dict) -> str:
#     # Replica el encabezado/estructura de _pp_result SIN la sección de evidencias
#     cands = res.get("retrieved_candidates") or []
#     reranked = res.get("reranked_hits") or []
#     answer = (res.get("answer") or "").strip()
#     query = res.get("query") or ""

#     return "\n".join([
#         "",  # _pp_result arranca imprimiendo un salto de línea antes del bloque
#         "=" * 88,
#         f"QUERY: {query}",
#         "-" * 88,
#         "ANSWER:\n",
#         answer,
#         "-" * 88,
#         f"Retrieved candidates: {len(cands)} | Reranked hits: {len(reranked)}",
#     ]).strip("\n")  # deja el formato limpio en celda / guardado

# def _extract_evidences(res: dict, max_hits: int = 10):
#     hits = res.get("reranked_hits") or res.get("retrieved_candidates") or []
#     out = []
#     for h in hits[:max_hits]:
#         file_date = h.get("file_date") or ""
#         path = h.get("path") or ""
#         page = h.get("page_num")
#         topic = h.get("topic") or ""
#         cid = h.get("chunk_id") or ""

#         loc_parts = []
#         if file_date: loc_parts.append(file_date)
#         if path: loc_parts.append(path)
#         if page is not None: loc_parts.append(f"p.{page}")
#         if topic: loc_parts.append(f"topic={topic}")
#         loc = " | ".join(loc_parts)

#         out.append({
#             "loc": loc,
#             "chunk_id": cid,
#             "file_date": file_date,
#             "path": path,
#             "page_num": page,
#             "topic": topic,
#         })
#     return out

# # ---------------------------
# # Run batch
# # ---------------------------
# if "mod" not in globals():
#     raise RuntimeError("No encuentro `mod` en el notebook. Ejecutá primero la celda donde cargás rag_agent.py (spec.loader.exec_module(mod)).")

# questions = _load_questions(QUESTIONS_FILE)

# rows = []
# for q in questions:
#     try:
#         res = mod.answer_with_rag(q)
#         respuesta_fmt = _format_answer_like_pp_result(res)
#         evidencias = _extract_evidences(res, max_hits=MAX_EVIDENCES_TO_STORE)
#         evidencias_json = json.dumps(evidencias, ensure_ascii=False)
#     except Exception as e:
#         respuesta_fmt = _format_answer_like_pp_result({"query": q, "answer": f"[ERROR] {type(e).__name__}: {e}", "retrieved_candidates": [], "reranked_hits": []})
#         evidencias_json = "[]"

#     rows.append(Row(Pregunta=q, Respuesta=respuesta_fmt, Evidencias=evidencias_json))

# df_out = spark.createDataFrame(rows)

# # ---------------------------
# # Save to Delta table (create if not exists, else truncate)
# # ---------------------------
# if spark.catalog.tableExists(TABLE_NAME):
#     spark.sql(f"TRUNCATE TABLE {TABLE_NAME}")
# else:
#     spark.sql(f"""
#       CREATE TABLE {TABLE_NAME} (
#         Pregunta  STRING,
#         Respuesta STRING,
#         Evidencias STRING
#       ) USING DELTA
#     """)

# df_out.write.format("delta").mode("append").saveAsTable(TABLE_NAME)

# display(df_out)


In [0]:
import re

def reemplazar_numeros_por_x(texto: str) -> str:
    """
    Reemplaza todos los valores numéricos en 'texto' por 'X'.
    Soporta enteros, decimales y negativos. Ej: "-12", "3.14", "1000"
    """
    patron = r'-?\d+(?:[.,]\d+)?'
    return re.sub(patron, "X", texto)


In [0]:
s = "<table><tr><th>Total</th><th>1.377.200</th><th>Banco CMF</th><th>19.398</th><th>Banco BICA</th><th>1.909</th><th>Banco Mariva</th><th>-3.600</th></tr><tr><td>Banco Santander</td><td>423.960</td><td>Banco de Servicios Financieros</td><td>15.377</td><td>Banco de Comercio</td><td>1.661</td><td>BiBank</td><td>-3.842</td></tr><tr><td>Banco Nación</td><td>283.637</td><td>Banco de Entre Rios</td><td>10.673</td><td>Banco Saenz</td><td>1.194</td><td>Banco Rioja</td><td>-8.280</td></tr><tr><td>Banco Macro</td><td>210.901</td><td>Banco de Santa Cruz</td><td>8.312</td><td>Banco Dino</td><td>694</td><td>Banco Columbia</td><td>-11.135</td></tr><tr><td>Banco Provincia</td><td>140.547</td><td>BST</td><td>8.216</td><td>Brubank</td><td>495</td><td>Banco Corrientes</td><td>-16.886</td></tr><tr><td>Citibank</td><td>80.141</td><td>Banco Piano</td><td>7.866</td><td>BNP Paribas</td><td>407</td><td>Banco Galicia</td><td>-17.924</td></tr><tr><td>Banco Ciudad</td><td>71.836</td><td>BBVA</td><td>7.118</td><td>Banco Meridian</td><td>400</td><td>Banco Supervielle</td><td>-18.458</td></tr><tr><td>Banco de Santiago del Estero</td><td>53.274</td><td>Banco Roela</td><td>7.087</td><td>BACS</td><td>65</td><td>Banco de Córdoba</td><td>-21.455</td></tr><tr><td>Banco del Neuquén</td><td>51.845</td><td>Banco de Santa Fe</td><td>6.920</td><td>Banco COINAG</td><td>-243</td><td>Uala Bank</td><td>-38.115</td></tr><tr><td>Banco Patagonia</td><td>40.593</td><td>BICE</td><td>5.960</td><td>RCI Banque</td><td>-1.070</td><td>CIB</td><td>-41.502</td></tr><tr><td>BIND Banco Industrial</td><td>37.396</td><td>Banco Comafi</td><td>4.066</td><td>Banco Julio</td><td>-1.098</td><td>Banco Credicop</td><td>-70.130</td></tr><tr><td>Banco de Valores</td><td>33.177</td><td>Banco Municipal de Rosario</td><td>4.011</td><td>Banco Masventas</td><td>-1.286</td><td></td><td></td></tr><tr><td>Nuevo Banco del Chaco</td><td>27.041</td><td>Banco de Tierra del Fuego</td><td>3.519</td><td>Bank of China</td><td>-1.441</td><td></td><td></td></tr><tr><td>Banco Misiones</td><td>21.780</td><td>Banco San Juan</td><td>3.196</td><td>Banco Sucrédito Regional</td><td>-1.448</td><td></td><td></td></tr><tr><td>Banco de Formosa</td><td>20.669</td><td>Banco del Sol</td><td>2.695</td><td>Banco Cetelerm</td><td>-2.173</td><td></td><td></td></tr><tr><td>JP Morgan Chase</td><td>20.397</td><td>Banco Chubut</td><td>2.394</td><td>Banco Rep. Or. del Uruguay</td><td>-2.220</td><td></td><td></td></tr><tr><td>BIND Banco Industrial</td><td>20.329</td><td>Banco de La Pampa</td><td>2.215</td><td>Banco Voi</td><td>-3.536</td><td></td><td></td></tr><tr><td>Banco General</td><td>19.860</td><td>Banco de Jujuy</td><td>2.150</td><td>Banco de Jujuy</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Galicia</td><td>19.760</td><td>Banco de Córdoba</td><td>2.140</td><td>Banco de Córdoba</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Itaú</td><td>19.750</td><td>Banco de Mendoza</td><td>2.130</td><td>Banco de Mendoza</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Mercantil</td><td>19.740</td><td>Banco de San Juan</td><td>2.120</td><td>Banco de San Juan</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Regional</td><td>19.730</td><td>Banco de Santa Fe</td><td>2.110</td><td>Banco de Santa Fe</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Santander</td><td>19.720</td><td>Banco de Tucumán</td><td>2.100</td><td>Banco de Tucumán</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.710</td><td>Banco de Ushuaia</td><td>2.090</td><td>Banco de Ushuaia</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.700</td><td>Banco de Viedma</td><td>2.080</td><td>Banco de Viedma</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.690</td><td>Banco de Zeballos</td><td>2.070</td><td>Banco de Zeballos</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.680</td><td>Banco de Zapla</td><td>2.060</td><td>Banco de Zapla</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.670</td><td>Banco de Zonas</td><td>2.050</td><td>Banco de Zonas</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.660</td><td>Banco de Zonas Rurales</td><td>2.040</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.650</td><td>Banco de Zonas Rurales</td><td>2.030</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.640</td><td>Banco de Zonas Rurales</td><td>2.020</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.630</td><td>Banco de Zonas Rurales</td><td>2.010</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.620</td><td>Banco de Zonas Rurales</td><td>2.000</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.610</td><td>Banco de Zonas Rurales</td><td>1.990</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.600</td><td>Banco de Zonas Rurales</td><td>1.980</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.590</td><td>Banco de Zonas Rurales</td><td>1.970</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.580</td><td>Banco de Zonas Rurales</td><td>1.960</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.570</td><td>Banco de Zonas Rurales</td><td>1.950</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.560</td><td>Banco de Zonas Rurales</td><td>1.940</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.550</td><td>Banco de Zonas Rurales</td><td>1.930</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.540</td><td>Banco de Zonas Rurales</td><td>1.920</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.530</td><td>Banco de Zonas Rurales</td><td>1.910</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.520</td><td>Banco de Zonas Rurales</td><td>1.900</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.510</td><td>Banco de Zonas Rurales</td><td>1.890</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.500</td><td>Banco de Zonas Rurales</td><td>1.880</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.490</td><td>Banco de Zonas Rurales</td><td>1.870</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.480</td><td>Banco de Zonas Rurales</td><td>1.860</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.470</td><td>Banco de Zonas Rurales</td><td>1.850</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.460</td><td>Banco de Zonas Rurales</td><td>1.840</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.450</td><td>Banco de Zonas Rurales</td><td>1.830</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.440</td><td>Banco de Zonas Rurales</td><td>1.820</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.430</td><td>Banco de Zonas Rurales</td><td>1.810</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.420</td><td>Banco de Zonas Rurales</td><td>1.800</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.410</td><td>Banco de Zonas Rurales</td><td>1.790</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.400</td><td>Banco de Zonas Rurales</td><td>1.780</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.390</td><td>Banco de Zonas Rurales</td><td>1.770</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.380</td><td>Banco de Zonas Rurales</td><td>1.760</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.370</td><td>Banco de Zonas Rurales</td><td>1.750</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.360</td><td>Banco de Zonas Rurales</td><td>1.740</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.350</td><td>Banco de Zonas Rurales</td><td>1.730</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.340</td><td>Banco de Zonas Rurales</td><td>1.720</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.330</td><td>Banco de Zonas Rurales</td><td>1.710</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.320</td><td>Banco de Zonas Rurales</td><td>1.700</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.310</td><td>Banco de Zonas Rurales</td><td>1.690</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.300</td><td>Banco de Zonas Rurales</td><td>1.680</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.290</td><td>Banco de Zonas Rurales</td><td>1.670</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.280</td><td>Banco de Zonas Rurales</td><td>1.660</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.270</td><td>Banco de Zonas Rurales</td><td>1.650</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.260</td><td>Banco de Zonas Rurales</td><td>1.640</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.250</td><td>Banco de Zonas Rurales</td><td>1.630</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.240</td><td>Banco de Zonas Rurales</td><td>1.620</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.230</td><td>Banco de Zonas Rurales</td><td>1.610</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.220</td><td>Banco de Zonas Rurales</td><td>1.600</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.210</td><td>Banco de Zonas Rurales</td><td>1.590</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.200</td><td>Banco de Zonas Rurales</td><td>1.580</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.190</td><td>Banco de Zonas Rurales</td><td>1.570</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.180</td><td>Banco de Zonas Rurales</td><td>1.560</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.170</td><td>Banco de Zonas Rurales</td><td>1.550</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.160</td><td>Banco de Zonas Rurales</td><td>1.540</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.150</td><td>Banco de Zonas Rurales</td><td>1.530</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.140</td><td>Banco de Zonas Rurales</td><td>1.520</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.130</td><td>Banco de Zonas Rurales</td><td>1.510</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.120</td><td>Banco de Zonas Rurales</td><td>1.500</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.110</td><td>Banco de Zonas Rurales</td><td>1.490</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.100</td><td>Banco de Zonas Rurales</td><td>1.480</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.090</td><td>Banco de Zonas Rurales</td><td>1.470</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.080</td><td>Banco de Zonas Rurales</td><td>1.460</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.070</td><td>Banco de Zonas Rurales</td><td>1.450</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.060</td><td>Banco de Zonas Rurales</td><td>1.440</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.050</td><td>Banco de Zonas Rurales</td><td>1.430</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.040</td><td>Banco de Zonas Rurales</td><td>1.420</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.030</td><td>Banco de Zonas Rurales</td><td>1.410</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.020</td><td>Banco de Zonas Rurales</td><td>1.400</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.010</td><td>Banco de Zonas Rurales</td><td>1.390</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>19.000</td><td>Banco de Zonas Rurales</td><td>1.380</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.990</td><td>Banco de Zonas Rurales</td><td>1.370</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.980</td><td>Banco de Zonas Rurales</td><td>1.360</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.970</td><td>Banco de Zonas Rurales</td><td>1.350</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.960</td><td>Banco de Zonas Rurales</td><td>1.340</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.950</td><td>Banco de Zonas Rurales</td><td>1.330</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.940</td><td>Banco de Zonas Rurales</td><td>1.320</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.930</td><td>Banco de Zonas Rurales</td><td>1.310</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.920</td><td>Banco de Zonas Rurales</td><td>1.300</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.910</td><td>Banco de Zonas Rurales</td><td>1.290</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.900</td><td>Banco de Zonas Rurales</td><td>1.280</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.890</td><td>Banco de Zonas Rurales</td><td>1.270</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.880</td><td>Banco de Zonas Rurales</td><td>1.260</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.870</td><td>Banco de Zonas Rurales</td><td>1.250</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.860</td><td>Banco de Zonas Rurales</td><td>1.240</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.850</td><td>Banco de Zonas Rurales</td><td>1.230</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.840</td><td>Banco de Zonas Rurales</td><td>1.220</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.830</td><td>Banco de Zonas Rurales</td><td>1.210</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.820</td><td>Banco de Zonas Rurales</td><td>1.200</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.810</td><td>Banco de Zonas Rurales</td><td>1.190</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.800</td><td>Banco de Zonas Rurales</td><td>1.180</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.790</td><td>Banco de Zonas Rurales</td><td>1.170</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.780</td><td>Banco de Zonas Rurales</td><td>1.160</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.770</td><td>Banco de Zonas Rurales</td><td>1.150</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.760</td><td>Banco de Zonas Rurales</td><td>1.140</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.750</td><td>Banco de Zonas Rurales</td><td>1.130</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.740</td><td>Banco de Zonas Rurales</td><td>1.120</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.730</td><td>Banco de Zonas Rurales</td><td>1.110</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.720</td><td>Banco de Zonas Rurales</td><td>1.100</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.710</td><td>Banco de Zonas Rurales</td><td>1.090</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.700</td><td>Banco de Zonas Rurales</td><td>1.080</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.690</td><td>Banco de Zonas Rurales</td><td>1.070</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.680</td><td>Banco de Zonas Rurales</td><td>1.060</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.670</td><td>Banco de Zonas Rurales</td><td>1.050</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.660</td><td>Banco de Zonas Rurales</td><td>1.040</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.650</td><td>Banco de Zonas Rurales</td><td>1.030</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.640</td><td>Banco de Zonas Rurales</td><td>1.020</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.630</td><td>Banco de Zonas Rurales</td><td>1.010</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.620</td><td>Banco de Zonas Rurales</td><td>1.000</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.610</td><td>Banco de Zonas Rurales</td><td>990</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.600</td><td>Banco de Zonas Rurales</td><td>980</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.590</td><td>Banco de Zonas Rurales</td><td>970</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.580</td><td>Banco de Zonas Rurales</td><td>960</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.570</td><td>Banco de Zonas Rurales</td><td>950</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.560</td><td>Banco de Zonas Rurales</td><td>940</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.550</td><td>Banco de Zonas Rurales</td><td>930</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.540</td><td>Banco de Zonas Rurales</td><td>920</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.530</td><td>Banco de Zonas Rurales</td><td>910</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.520</td><td>Banco de Zonas Rurales</td><td>900</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.510</td><td>Banco de Zonas Rurales</td><td>890</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.500</td><td>Banco de Zonas Rurales</td><td>880</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.490</td><td>Banco de Zonas Rurales</td><td>870</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.480</td><td>Banco de Zonas Rurales</td><td>860</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.470</td><td>Banco de Zonas Rurales</td><td>850</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.460</td><td>Banco de Zonas Rurales</td><td>840</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.450</td><td>Banco de Zonas Rurales</td><td>830</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.440</td><td>Banco de Zonas Rurales</td><td>820</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.430</td><td>Banco de Zonas Rurales</td><td>810</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.420</td><td>Banco de Zonas Rurales</td><td>800</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.410</td><td>Banco de Zonas Rurales</td><td>790</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.400</td><td>Banco de Zonas Rurales</td><td>780</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.390</td><td>Banco de Zonas Rurales</td><td>770</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.380</td><td>Banco de Zonas Rurales</td><td>760</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.370</td><td>Banco de Zonas Rurales</td><td>750</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.360</td><td>Banco de Zonas Rurales</td><td>740</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.350</td><td>Banco de Zonas Rurales</td><td>730</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.340</td><td>Banco de Zonas Rurales</td><td>720</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.330</td><td>Banco de Zonas Rurales</td><td>710</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr><tr><td>Banco Credicop</td><td>18.320</td><td>Banco de Zonas Rurales</td><td>700</td><td>Banco de Zonas Rurales</td><td>-3.610</td><td></td><td></td></tr>"
print(reemplazar_numeros_por_x(s))
# "Tengo X perros, gasté X pesos y caminé X km."
